In [1]:
from collections import defaultdict

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

/home/randy/Documents/ACompArc/model/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_ID = "roneneldan/TinyStories-3M"

# Pin the exact model revision for reproducible experiments.
MODEL_REVISION = "cfaf26ec85ecdfc1bd7c2638104cce55cb67f894"

In [3]:
print(f"PyTorch version: {torch.__version__}")
print(f"Loading {MODEL_ID}...")

PyTorch version: 2.13.0+cu130
Loading roneneldan/TinyStories-3M...


In [4]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
)

W0922 21:12:10.060000 11849 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0922 21:12:10.090000 11849 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Loading weights: 100%|██████████| 108/108 [00:00<00:00, 26048.58it/s]
[transformers] GPTNeoForCausalLM LOAD REPORT from: roneneldan/TinyStories-3M
Key                                                               | Status     |  | 
------------------------------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5, 6, 7}.attn.attention.masked_bias | UNEXPECTED |  | 
transformer.h.{0, 1, 2, 3, 4, 5, 6, 7}.a

In [5]:
model.eval()

GPTNeoForCausalLM(
  (transformer): GPTNeoModel(
    (wte): Embedding(50257, 128)
    (wpe): Embedding(2048, 128)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-7): 8 x GPTNeoBlock(
        (ln_1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPTNeoAttention(
          (attention): GPTNeoSelfAttention(
            (attn_dropout): Dropout(p=0.0, inplace=False)
            (resid_dropout): Dropout(p=0.0, inplace=False)
            (k_proj): Linear(in_features=128, out_features=128, bias=False)
            (v_proj): Linear(in_features=128, out_features=128, bias=False)
            (q_proj): Linear(in_features=128, out_features=128, bias=False)
            (out_proj): Linear(in_features=128, out_features=128, bias=True)
          )
        )
        (ln_2): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPTNeoMLP(
          (c_fc): Linear(in_features=128, out_features=512, bias=True)
          (c_

In [6]:
# model.parameters() counts independent Parameter objects.
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

parameter_bytes = sum(
    parameter.numel() * parameter.element_size()
    for parameter in model.parameters()
)

In [7]:
print("\n=== Model type ===")
print(type(model).__name__)


=== Model type ===
GPTNeoForCausalLM


In [8]:
print("\n=== Configuration ===")
print(f"Vocabulary size        : {model.config.vocab_size:,}")
print(f"Maximum sequence length: {model.config.max_position_embeddings:,}")
print(f"Transformer layers     : {model.config.num_layers}")
print(f"Hidden dimension       : {model.config.hidden_size}")
print(f"Attention heads        : {model.config.num_heads}")
print(f"Attention types        : {model.config.attention_layers}")
print(f"Activation function    : {model.config.activation_function}")


=== Configuration ===
Vocabulary size        : 50,257
Maximum sequence length: 2,048
Transformer layers     : 8
Hidden dimension       : 128
Attention heads        : 16
Attention types        : ['global', 'local', 'global', 'local', 'global', 'local', 'global', 'local']
Activation function    : gelu_new


In [9]:
print("\n=== Parameter count ===")
print(f"Total parameters       : {total_parameters:,}")
print(f"Trainable parameters   : {trainable_parameters:,}")
print(f"FP32 parameter storage : {parameter_bytes / (1024**2):.2f} MiB")
print(f"Estimated INT8 storage : {total_parameters / (1024**2):.2f} MiB")


=== Parameter count ===
Total parameters       : 8,278,400
Trainable parameters   : 8,278,400
FP32 parameter storage : 31.58 MiB
Estimated INT8 storage : 7.89 MiB


In [10]:
# Count parameters by major model component.
component_counts = defaultdict(int)

for name, parameter in model.named_parameters():
    parts = name.split(".")

    if len(parts) >= 3 and parts[0] == "transformer" and parts[1] == "h":
        component = f"transformer_block_{parts[2]}"
    elif name.startswith("transformer.wte"):
        component = "token_embedding"
    elif name.startswith("transformer.wpe"):
        component = "position_embedding"
    elif name.startswith("transformer.ln_f"):
        component = "final_layer_norm"
    elif name.startswith("lm_head"):
        component = "language_model_head"
    else:
        component = "other"

    component_counts[component] += parameter.numel()

In [11]:
print("\n=== Parameters by component ===")

for component, count in component_counts.items():
    percentage = 100.0 * count / total_parameters
    print(f"{component:24s}: {count:10,d} ({percentage:6.2f}%)")


=== Parameters by component ===
token_embedding         :  6,432,896 ( 77.71%)
position_embedding      :    262,144 (  3.17%)
transformer_block_0     :    197,888 (  2.39%)
transformer_block_1     :    197,888 (  2.39%)
transformer_block_2     :    197,888 (  2.39%)
transformer_block_3     :    197,888 (  2.39%)
transformer_block_4     :    197,888 (  2.39%)
transformer_block_5     :    197,888 (  2.39%)
transformer_block_6     :    197,888 (  2.39%)
transformer_block_7     :    197,888 (  2.39%)
final_layer_norm        :        256 (  0.00%)


In [12]:
# GPT-style models commonly share the token-embedding matrix with
# the output language-model head.
input_embedding = model.get_input_embeddings().weight
output_embedding = model.get_output_embeddings().weight

weights_are_tied = (
    input_embedding.data_ptr() == output_embedding.data_ptr()
)

print("\n=== Weight sharing ===")
print(f"Input/output embeddings tied: {weights_are_tied}")


=== Weight sharing ===
Input/output embeddings tied: True
